# Linear Search and Sequential Scanning

**Session:** 2026-10-28 · Brute-force Exhaustive  
**Module:** 01 — Python Fundamentals & Algorithmic Thinking

The **brute force** paradigm solves computational problems by applying definitions directly without heuristics or shortcuts. The most fundamental brute-force search technique is **Linear Search** (sequential scan), which examines elements one by one until a match is found or the collection is exhausted.

## Learning Outcomes

- Implement linear search on unsorted and sorted sequences.
- Optimize sequential search using the sentinel technique to reduce boundary checks.
- Formulate line-by-line cost-and-frequency tables and derive Best, Average, and Worst case complexities.
- Connect linear scan to exact nearest neighbor search in Machine Learning.


## 1. Standard Linear Search

Given a sequence $\langle a_0, a_1, \dots, a_{n-1} \rangle$ and a target `key`, inspect each index $i$ from $0$ to $n-1$. Return $i$ upon first match, or $-1$ if absent.


In [ ]:
from collections.abc import Sequence
from typing import Any


def linear_search(array: Sequence[Any], key: Any) -> int:
    """Perform sequential scan to find the first occurrence of key.

    Args:
        array: Sequence of elements to inspect.
        key: Target value to search for.

    Returns:
        Index of the first match, or -1 if not found.
    """
    for index, element in enumerate(array):
        if element == key:
            return index
    return -1


# Verification on lecture test cases
sample_array = [1, 2, 10, 15, 23]
assert linear_search(sample_array, 23) == 4
assert linear_search(sample_array, 1) == 0
assert linear_search(sample_array, 10) == 2
assert linear_search(sample_array, 34) == -1
print("Standard linear search verified successfully!")

## 2. Cost Analysis and Operation Counting

Let $n$ be the length of the array:

| Line / Operation | Cost ($c_i$) | Best Case (Key at index 0) | Worst Case (Key absent) | Average Case (Uniform random) |
| :--- | :--- | :--- | :--- | :--- |
| Loop index check (`i < n`) | $c_1$ | $1$ | $n + 1$ | $\frac{n+1}{2}$ |
| Element lookup (`array[i]`) | $c_2$ | $1$ | $n$ | $\frac{n+1}{2}$ |
| Equality check (`== key`) | $c_3$ | $1$ | $n$ | $\frac{n+1}{2}$ |
| Return statement | $c_4$ | $1$ | $1$ | $1$ |

- **Best Case:** $O(1)$ comparisons when `key == array[0]`.
- **Worst Case:** $O(n)$ comparisons when `key` is at the last index or absent.
- **Average Case:** Assuming uniform distribution over $n$ positions, expected comparisons:
  $$\mathbb{E}[C] = \sum_{i=1}^n \frac{1}{n} \cdot i = \frac{1}{n} \frac{n(n+1)}{2} = \frac{n+1}{2} = O(n)$$


In [ ]:
def instrumented_linear_search(array: Sequence[Any], key: Any) -> tuple[int, int]:
    """Linear search tracking exact comparison count."""
    comparisons = 0
    for idx, item in enumerate(array):
        comparisons += 1
        if item == key:
            return idx, comparisons
    return -1, comparisons


# Step-by-step verification from lecture slides
data = [1, 2, 10, 15, 23]
queries = [(1, 1), (10, 3), (23, 5), (34, 5)]

for query_key, expected_ops in queries:
    idx, ops = instrumented_linear_search(data, query_key)
    print(
        f"Key {query_key:2d}: found at index {idx:2d} after {ops} comparisons (expected {expected_ops})"
    )
    assert ops == expected_ops

## 3. Sentinel Linear Search Optimization

In standard linear search, each iteration performs **two** conditional evaluations:
1. Loop termination check: `i < n`
2. Equality check: `array[i] == key`

By temporarily placing `key` at the end of the array as a **sentinel**, the loop termination condition can be omitted entirely, reducing loop overhead by ~50% in compiled languages.


In [ ]:
def sentinel_linear_search(array: list[Any], key: Any) -> int:
    """Linear search using a sentinel value to eliminate bounds checking."""
    n = len(array)
    if n == 0:
        return -1

    last_val = array[-1]
    if last_val == key:
        return n - 1

    array[-1] = key
    i = 0
    while array[i] != key:
        i += 1

    # Restore mutated tail
    array[-1] = last_val

    if i < n - 1 or last_val == key:
        return i
    return -1


numbers = [7, 2, 5, 4, 17]
assert sentinel_linear_search(numbers.copy(), 5) == 2
assert sentinel_linear_search(numbers.copy(), 17) == 4
assert sentinel_linear_search(numbers.copy(), 99) == -1
print("Sentinel linear search verified successfully!")

## 4. Machine Learning Application: Flat Index Exact KNN Search

In information retrieval and embedding search, brute force linear scanning over database vectors is known as a **Flat Index**. It computes pairwise distance to all $N$ vectors:
- **Time complexity:** $O(N \cdot D)$ where $D$ is embedding dimension.
- **Accuracy:** 100% recall (ground truth for evaluating ANN methods like HNSW, IVF).


In [ ]:
import math


def flat_knn_search(
    database: list[list[float]], query: list[float], k: int = 1
) -> list[tuple[int, float]]:
    """Exhaustive sequential scan to find k nearest neighbors by Euclidean distance."""
    scored = []
    for idx, vec in enumerate(database):
        dist_sq = sum((x - q) ** 2 for x, q in zip(vec, query, strict=True))
        scored.append((idx, math.sqrt(dist_sq)))

    scored.sort(key=lambda item: item[1])
    return scored[:k]


# Test flat exact search
db = [[0.1, 0.2, 0.9], [0.8, 0.9, 0.1], [0.15, 0.25, 0.85], [0.9, 0.8, 0.2]]
q = [0.12, 0.22, 0.88]
top1 = flat_knn_search(db, q, k=1)
print(f"Nearest neighbor vector index: {top1[0][0]}, distance: {top1[0][1]:.4f}")
assert top1[0][0] in (0, 2)

## Summary

- Linear search embodies the brute-force search pattern: $O(1)$ best case, $O(n)$ average and worst case.
- Sentinel search optimizes execution constant factors without changing asymptotic order.
- In machine learning, exhaustive sequential search provides the exact baseline against which all approximate indexing algorithms are validated.
